# Project 2 - Poker Hand Prediction by Classification

이 노트북은 제출 코드의 핵심 구현과 실행 결과를 정리한 파일입니다.

- `hand_features.py`의 카드 특징 추출 함수
- `train.py`의 3장/4장 카드 분류 모델 학습
- 테스트 데이터 평가
- `poker.py`에서 사용하는 `predict_call()` 구현

실행할 때는 학습·테스트 CSV 파일과 이 노트북을 같은 폴더에 둡니다.

## 1. 라이브러리 불러오기

In [1]:
import pickle
import pandas as pd
from collections import Counter
from sklearn.tree import DecisionTreeClassifier

## 2. 카드 특징 추출 함수

In [2]:
STRAIGHTS = [set(range(start, start + 5)) for start in range(1, 10)]
STRAIGHTS.append({10, 11, 12, 13, 1})

def describe_hand(cards):
    suits, ranks = zip(*cards)
    groups = sorted(Counter(ranks).values(), reverse=True)
    groups += [0] * (4 - len(groups))
    unique_ranks = set(ranks)
    matching_suits = max(Counter(suits).values())
    possible_straights = sum(unique_ranks <= run for run in STRAIGHTS)
    return [
        *groups, matching_suits, possible_straights,
        int(unique_ranks <= STRAIGHTS[-1]),
    ]

## 3. 학습 데이터 불러오기 및 모델 학습
3장 카드와 4장 카드 상황을 각각 학습하고 모델 파일로 저장합니다.

In [3]:
data = pd.read_csv("poker-hand-training.csv")
labels = data.iloc[:, -1]

for visible in (3, 4):
    rows = data.iloc[:, :visible * 2].itertuples(index=False, name=None)
    features = []
    for row in rows:
        cards = list(zip(row[::2], row[1::2]))
        features.append(describe_hand(cards))

    model = DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=16,
        random_state=17
    )
    model.fit(features, labels)

    filename = f"poker-model{visible}.sav"
    with open(filename, "wb") as file:
        pickle.dump(model, file, protocol=4)
    print(f"{filename} saved.")

poker-model3.sav saved.
poker-model4.sav saved.


## 4. 테스트 데이터로 모델 평가

In [4]:
test_data = pd.read_csv("poker-hand-testing.csv")
test_labels = test_data.iloc[:, -1]

for visible in (3, 4):
    rows = test_data.iloc[:, :visible * 2].itertuples(index=False, name=None)
    test_features = [
        describe_hand(list(zip(row[::2], row[1::2])))
        for row in rows
    ]

    with open(f"poker-model{visible}.sav", "rb") as file:
        model = pickle.load(file)

    predictions = model.predict(test_features)
    accuracy = (predictions == test_labels).sum() / test_labels.count()
    print(f"{visible}-card test accuracy: {accuracy:.4f}")

3-card test accuracy: 0.6302
4-card test accuracy: 0.7739


## 5. 게임에서 사용하는 CALL 판단 함수

카드 수에 맞는 모델을 선택하고, 예측 확률로 포커 점수의 기댓값을 계산합니다. 기댓값이 상대방 점수보다 높으면 `True`(CALL), 아니면 `False`(FOLD)를 반환합니다.

In [5]:
with open("poker-model3.sav", "rb") as file:
    model3 = pickle.load(file)
with open("poker-model4.sav", "rb") as file:
    model4 = pickle.load(file)

def predict_call(hand, opposite_score):
    if len(hand) == 3:
        model = model3
    else:
        model = model4

    features = describe_hand(hand)
    probabilities = model.predict_proba([features])[0]

    expected_score = 0
    for score, probability in zip(model.classes_, probabilities):
        expected_score += score * probability

    return expected_score > opposite_score

## 6. 구현 정리

- `describe_hand()`는 카드 정보를 모델에 넣을 특징으로 변환합니다.
- `DecisionTreeClassifier`로 3장용과 4장용 모델을 따로 학습합니다.
- 모델은 `.sav` 파일로 저장합니다.
- `predict_call()`은 예측 확률로 계산한 점수의 기댓값을 상대방 점수와 비교합니다.

실제 Pygame 게임 화면은 별도 파일 `poker.py`에서 실행합니다.

## 7. 전체 게임 코드 (`poker.py`)

아래는 별도 제출 파일 `poker.py`의 전체 코드입니다. Pygame 게임을 실행하는 코드이므로 노트북 안에서는 자동 실행하지 않고, 코드 확인용으로 포함합니다.

In [ ]:
from pydoc import cli
import pygame, sys
pygame.init()

# NOTE: The following implementation is a sample and will not run without 
# the necessary model files (e.g., SAV files).
## ---------------- FILL HERE ---------------------------------
import pickle
from hand_features import describe_hand

# 학습된 3장 및 4장 카드 분류 모델 불러오기
with open("poker-model3.sav", "rb") as file:
    model3 = pickle.load(file)

with open("poker-model4.sav", "rb") as file:
    model4 = pickle.load(file)


def predict_call(hand, opposite_score):
    # 공개된 카드 수에 맞는 모델 선택
    if len(hand) == 3:
        model = model3
    else:
        model = model4

    # 학습할 때 사용한 것과 같은 특징 추출
    features = describe_hand(hand)

    # 각 포커 점수에 대한 예측 확률
    probabilities = model.predict_proba([features])[0]

    # 예측 확률을 이용해 포커 점수의 기댓값 계산
    expected_score = 0
    for score, probability in zip(model.classes_, probabilities):
        expected_score += score * probability

    # 상대방 점수보다 높은 기댓값이면 CALL
    return expected_score > opposite_score
## -------------------------------------------------

import pandas as pd
cand_hands = pd.read_csv('poker-hand-testing.csv')

def pick_opposite_score():
    return cand_hands['Poker Hand'].sample().values[0]

def pick_five_cards():
    sam = cand_hands.sample().values[0]
    score = sam[-1]
    lis = sam[:-1]
    return (
        [ (a, b) for a, b in zip(lis[::2], lis[1::2]) ],
        score)


# game money: initially, $1000
money = 1000

# game count
#game_count = 1000

# initially, pick five cards
full_hand, hand_score = pick_five_cards()
print(full_hand, hand_score)

# currently, three flipped cards
flipped = 3
current_cards = full_hand[:flipped]

# current game's opposite hand score
opposite_hand_score = pick_opposite_score()

#game_count = game_count - 1
money = money - 1

font = pygame.font.SysFont("comicsans",20)
SCREEN_WIDTH, SCREEN_HEIGHT = 512, 512

# create window
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("POKER")

CARD_WIDTH, CARD_HEIGHT = 60, 100
SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT = 32, 64

# create a surface object, image is drawn on it.
imp = pygame.image.load("cards.jpg")
imp = pygame.transform.scale(imp, (SUIT_ICON_WIDTH*4, SUIT_ICON_HEIGHT))

# list of suit icons (0: hear, 1: spade, 2: diamond, 3: club)
suits = list()

# 0: heart
surf = pygame.Surface((SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT))
surf.blit(imp, (0, 0), (SUIT_ICON_WIDTH*3, 0, SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT) )
suits.append(surf)

# 1: spade
surf = pygame.Surface((SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT))
surf.blit(imp, (0, 0), (0, 0, SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT) )
suits.append(surf)

# 2: diamond
surf = pygame.Surface((SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT))
surf.blit(imp, (0, 0), (SUIT_ICON_WIDTH, 0, SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT) )
suits.append(surf)

# 3: club
surf = pygame.Surface((SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT))
surf.blit(imp, (0, 0), (SUIT_ICON_WIDTH*2, 0, SUIT_ICON_WIDTH, SUIT_ICON_HEIGHT) )
suits.append(surf)

# colors
white = (255, 255, 255)
black = (0, 0, 0)
gray = (128, 128, 128)

def draw_card(suit, rank, pos):
    # white card
    rect = pygame.Rect( pos, (CARD_WIDTH, CARD_HEIGHT) )
    pygame.draw.rect(screen, white, rect)

    # suit icon
    screen.blit(
        suits[suit], 
        (pos[0] + CARD_WIDTH/2 - SUIT_ICON_WIDTH/2, 
            pos[1] + CARD_HEIGHT*2/3 - SUIT_ICON_HEIGHT/2))

    # rank
    rank_text = font.render("{}".format(rank), True, black)
    rank_rect = rank_text.get_rect()
    rank_rect.center = (pos[0] + CARD_WIDTH/2, pos[1] + CARD_HEIGHT/4)
    screen.blit(rank_text, rank_rect)

def gamemoney_text():
    text = font.render("Your money: ${}".format(money), True, white)
    rank_rect = text.get_rect()
    rank_rect.center = (SCREEN_WIDTH/2, SCREEN_HEIGHT/6)
    screen.blit(text, rank_rect)

def opposite_hand_text():
    text = font.render(
        "The opposite's hand score: {}".format(opposite_hand_score), 
        True, white)
    rank_rect = text.get_rect()
    rank_rect.center = (SCREEN_WIDTH/2, SCREEN_HEIGHT/6 + rank_rect.height)
    screen.blit(text, rank_rect)

BUTTON_WIDTH, BUTTON_HEIGHT = 120, 50

def draw_button(center_pos, button_text, button_color):
    # rect button
    #rect = pygame.Rect(
    #    (center_pos[0] - BUTTON_WIDTH/2, center_pos[1] - BUTTON_HEIGHT/2),
    #    (BUTTON_WIDTH, BUTTON_HEIGHT) )
    rect = pygame.Rect((0, 0, BUTTON_WIDTH, BUTTON_HEIGHT))
    rect.center = center_pos
    pygame.draw.rect(screen, button_color, rect) 

    # button text
    text = font.render(button_text, True, black)
    text_rect = text.get_rect()
    text_rect.center = rect.center
    screen.blit(text, text_rect)

msg = None
MESSAGE_HEIGHT = 40
def display_message(_msg):
    if _msg is not None:
        # white message board at the top of screen
        rect = pygame.Rect( (0, 0), (SCREEN_WIDTH, MESSAGE_HEIGHT) )
        pygame.draw.rect(screen, white, rect)
        
        # display message text
        text = font.render(_msg, True, black)
        rect = text.get_rect()
        rect.center = (SCREEN_WIDTH/2, MESSAGE_HEIGHT/2)
        screen.blit(text, rect)

# bet and die buttons at the bottom of screen
BET_BUTTON_CENTER = (SCREEN_WIDTH/4, SCREEN_HEIGHT*4/5)
bet_button_color = white

def draw_bet_button():
    draw_button(BET_BUTTON_CENTER, "CALL", bet_button_color)

# answer if pos is on the button
def on_bet_button(pos):
    return (abs(pos[0] - BET_BUTTON_CENTER[0]) <= BUTTON_WIDTH/2) \
        and (abs(pos[1] - BET_BUTTON_CENTER[1]) <= BUTTON_HEIGHT/2)

DIE_BUTTON_CENTER = (SCREEN_WIDTH/2, SCREEN_HEIGHT*4/5)
die_button_color = white

def draw_die_button():
    draw_button(DIE_BUTTON_CENTER, "FOLD", die_button_color)

def on_die_button(pos):
    return (abs(pos[0] - DIE_BUTTON_CENTER[0]) <= BUTTON_WIDTH/2) \
        and (abs(pos[1] - DIE_BUTTON_CENTER[1]) <= BUTTON_HEIGHT/2)

# auto play with classifier
CLS_BUTTON_CENTER = (SCREEN_WIDTH*3/4, SCREEN_HEIGHT*4/5)
cls_button_color = white

def draw_cls_button():
    draw_button(CLS_BUTTON_CENTER, "AUTO", cls_button_color)

def on_cls_button(pos):
    return (abs(pos[0] - CLS_BUTTON_CENTER[0]) <= BUTTON_WIDTH/2) \
        and (abs(pos[1] - CLS_BUTTON_CENTER[1]) <= BUTTON_HEIGHT/2)

# play sound when cards are revealed
flip_sound = pygame.mixer.Sound("flipcard.wav")
def play_cardflip_sound(rep):
    pygame.mixer.Sound.play(flip_sound, loops=rep-1)

def draw_current_cards():
    for i, (suit, rank) in enumerate(current_cards):
        draw_card(
            suit-1, rank, # suit and rank 
            ((i+1)*SCREEN_WIDTH/7, SCREEN_HEIGHT*2/5))

# game state for updating screen
# 0: beginning of game - three cards are flipped
# 1: bet - one more card is flipped
# 2: bet & game over - the last card is flipped and
#    win/tie/lose messages are shown
#    then, go to the next game after 5 secs
# 3: die & game over - message shown then,
#    go to the next game after 5 secs
# 9: wait for the next action
update_state = 0

def draw_items_on_screen():
    screen.fill(black)
    opposite_hand_text()
    gamemoney_text()
    draw_bet_button()
    draw_die_button()
    draw_cls_button()
    draw_current_cards()

    if msg is not None:
        display_message(msg)

    pygame.display.flip()
    pygame.time.delay(200)

# drawing cards and playing sound
def update_cards_and_sounds():
    global flipped, update_state, full_hand, hand_score,\
        money, msg, current_cards, opposite_hand_score

    if update_state == 0:
        # at the beginning of game, three cards are flipped
        play_cardflip_sound(3)
        draw_items_on_screen()

        # then, the screen of waiting the next action
        update_state = 9

    elif update_state == 1:
        # check button clicked and now four cards revealed
        play_cardflip_sound(1)
        draw_items_on_screen()

        # then, the screen of waiting the next action
        update_state = 9

    elif update_state == 2 or update_state == 3:
        # the last cards are revealed
        play_cardflip_sound(1)
        draw_items_on_screen()

        # delay another 1 sec for showing message
        pygame.time.delay(1000)
        msg = None
        draw_items_on_screen()

        # a new poker hand is drawn
        full_hand, hand_score = pick_five_cards()
        print(full_hand, hand_score)

        # three flipped cards
        flipped = 3
        current_cards = full_hand[:flipped]

        # opposite's hand score
        opposite_hand_score = pick_opposite_score()
        money = money - 1
        update_state = 0


game_stop = False
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN and not game_stop:
            click_pos = pygame.mouse.get_pos()

            auto_bet = False
            auto_die = False
            if on_cls_button(click_pos):
                if predict_call(current_cards, opposite_hand_score):
                    print("auto call!")
                    auto_bet = True
                    display_message("AI calls")
                    pygame.display.update()
                    pygame.time.delay(800)
                else:
                    print("auto fold!")
                    auto_die = True
                    display_message("AI folds")
                    pygame.display.update()
                    pygame.time.delay(800)

            if on_bet_button(click_pos) or auto_bet:
                flipped = flipped + 1
                current_cards = full_hand[:flipped]
                money = money - 1

                # if game over,
                if flipped == 5:
                    # case 1: win
                    if hand_score > opposite_hand_score:
                        money = money + 6
                        msg = "Win"
                        update_state = 2
                    # case 2: tie
                    elif hand_score == opposite_hand_score:
                        money = money + 3
                        msg = "Tie"
                        update_state = 2
                    # case 3: lose
                    else:
                        msg = "Lose"
                        update_state = 2
                # if cards can be flipped more,
                else:
                    update_state = 1
                        
            elif on_die_button(click_pos) or auto_die:
                msg = "Fold, go on to the next"
                update_state = 3

    # show cards and info message including sound
    update_cards_and_sounds()

    pygame.time.delay(200)